# Random QUBO (n=20) vs Concatenated Random QUBO (4×5) SA 비교 실험

**목적**: 동일 크기(n=20)의 random QUBO에서 구조적 차이가 SA 난이도에 미치는 영향 비교

| 조건 | 설명 |
|------|------|
| **Single (n=20)** | 20×20 fully-connected random QUBO. brute force로 GS 탐색 |
| **Concat (4×5)** | 5×5 random QUBO 4개를 block-diagonal로 접합. 각 블록별 brute force로 GS 탐색 |

- 계수: `np.random.uniform(-1, 1)` 연속 균일분포
- 판정: Energy 성공률 (`|SA_energy - target_energy| < tol`)

In [6]:
import numpy as np
from itertools import product
import random
import sys
import neal
import time
import json
import os

np.set_printoptions(
    threshold=sys.maxsize,
    linewidth=150,
    precision=3,
    suppress=True
)

## 공통 함수 정의

In [7]:
def gen_random_qubo(n):
    """n x n 랜덤 상삼각 QUBO 생성 (연속 균일분포 U(-1,1))"""
    random_qubo = np.random.uniform(-1, 1, (n, n))
    return np.triu(random_qubo)

def find_opt_brute_force(mat, debug=False):
    """brute force로 최적해, 최적 값, 축퇴도 탐색"""
    n = mat.shape[0]
    best_x = None
    best_val = float('inf')
    num_degenerate = 0

    for bits in product([0, 1], repeat=n):
        x = np.array(bits)
        cur_val = x @ mat @ x
        if cur_val < best_val - 1e-12:
            best_val = cur_val
            best_x = x
            num_degenerate = 1
        elif abs(cur_val - best_val) < 1e-12:
            num_degenerate += 1

    if debug:
        print("opt_x:", best_x, "\nopt_val:", best_val, "\ndegeneracy:", num_degenerate)
    return best_x, best_val, num_degenerate

def gen_concatenated_random_qubo(n, max_sub_graph_size, debug=False):
    """block-diagonal 균등 분할 random QUBO 생성"""
    k = max(1, -(-n // max_sub_graph_size))  # ceil(n / max_size)
    mat = np.zeros((n, n))
    opt = np.zeros(n)

    for i in range(k):
        start = i * n // k
        end = (i + 1) * n // k
        cur_n = end - start

        cur_mat = gen_random_qubo(cur_n)
        cur_opt, _, _ = find_opt_brute_force(cur_mat)

        mat[start:end, start:end] = cur_mat
        opt[start:end] = cur_opt

        if debug:
            print(f"  partition {i}: vars [{start}, {end}), size={cur_n}")

    if debug:
        print(f"  총 {k}개 partition, 크기: {[((i+1)*n//k - i*n//k) for i in range(k)]}")
    return mat, opt

def matrix_to_qubo_dict(mat):
    """numpy 행렬 → dict 변환 (neal 입력 형식)"""
    Q = {}
    n = mat.shape[0]
    for i in range(n):
        for j in range(i, n):
            if mat[i][j] != 0:
                Q[(i, j)] = float(mat[i][j])
    return Q

## 생성 예시 확인

Single n=20 vs Concat 4×5 구조 비교

In [8]:
# Single n=20: fully-connected random QUBO
mat_single = gen_random_qubo(20)
opt_single, val_single, deg_single = find_opt_brute_force(mat_single)
print("=== Single n=20 ===")
print("opt_x:", opt_single)
print("opt_val:", val_single, "  degeneracy:", deg_single)
print("non-zero entries:", np.count_nonzero(mat_single))
print()

# Concat 4×5: block-diagonal
mat_concat, opt_concat = gen_concatenated_random_qubo(20, 5, debug=True)
val_concat = opt_concat @ mat_concat @ opt_concat
_, _, deg_concat = find_opt_brute_force(mat_concat)
print("opt_x:", opt_concat)
print("opt_val:", val_concat, "  degeneracy:", deg_concat)
print("non-zero entries:", np.count_nonzero(mat_concat))

=== Single n=20 ===
opt_x: [1 0 1 0 1 0 1 0 0 0 0 1 0 1 1 0 0 1 0 1]
opt_val: -14.057819289582948   degeneracy: 1
non-zero entries: 210

  partition 0: vars [0, 5), size=5
  partition 1: vars [5, 10), size=5
  partition 2: vars [10, 15), size=5
  partition 3: vars [15, 20), size=5
  총 4개 partition, 크기: [5, 5, 5, 5]
opt_x: [0. 0. 0. 1. 0. 1. 1. 1. 0. 1. 1. 0. 0. 1. 1. 1. 1. 1. 1. 0.]
opt_val: -8.350103802206696   degeneracy: 1
non-zero entries: 60


## SA 실험: Single (n=20) vs Concat (4×5)

각 인스턴스에서:
1. **Single**: `gen_random_qubo(20)` → `find_opt_brute_force` → SA
2. **Concat**: `gen_concatenated_random_qubo(20, 5)` → SA

**축퇴 처리**: `lin2`에서는 축퇴(동일 에너지 GS 여러 개)가 빈번함
- Energy 성공률: `|SA_energy - target_energy| < tol` (축퇴 무관, 에너지만 비교)
- Bit 성공률: SA 결과가 brute force로 찾은 target과 정확히 일치 (축퇴 시 낮을 수 있음)

In [9]:
# ─── 하이퍼파라미터 ───
num_instances = 500
num_reads = 100
num_sweeps = 1000
energy_tol = 1e-6

sa_n = 20
sa_max_sub = 5  # concat: 4 blocks of 5

# ─── 결과 저장 경로 ───
results_dir = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'results')
os.makedirs(results_dir, exist_ok=True)
results_path = os.path.join(results_dir,
    f'single_vs_concat_n{sa_n}_sub{sa_max_sub}_uniform_inst{num_instances}_sw{num_sweeps}.json')

sampler = neal.SimulatedAnnealingSampler()

# ─── 통계 ───
conditions = ['single_n20', 'concat_4x5']
stats = {c: {'eng_ok': 0, 'bit_ok': 0, 'hd_sum': 0.0, 'deg_sum': 0, 'total': 0} for c in conditions}
all_results = []

print(f"═══ SA 실험: Single (n=20) vs Concat (4×5) ═══")
print(f"  n={sa_n}, max_sub={sa_max_sub}, coeff=uniform(-1,1)")
print(f"  instances={num_instances}, reads={num_reads}, sweeps={num_sweeps}")
print(f"  energy_tol={energy_tol}")
print(f"  저장 경로: {results_path}")
print()

t0 = time.perf_counter()

for inst in range(num_instances):
    inst_result = {'inst': inst}

    # ── 1. Single n=20 ──
    mat_s = gen_random_qubo(sa_n)
    opt_s, val_s, deg_s = find_opt_brute_force(mat_s)
    target_s = ''.join(str(int(x)) for x in opt_s)

    Q_dict_s = matrix_to_qubo_dict(mat_s)
    ss_s = sampler.sample_qubo(Q_dict_s, num_reads=num_reads, num_sweeps=num_sweeps)

    eng_ok_s = 0
    bit_ok_s = 0
    hd_sum_s = 0
    for sample, energy, _ in ss_s.data(['sample', 'energy', 'num_occurrences']):
        found = ''.join(str(sample[k]) for k in range(sa_n))
        hd = sum(1 for a, b in zip(target_s, found) if a != b)
        hd_sum_s += hd
        if found == target_s:
            bit_ok_s += 1
        if abs(energy - val_s) < energy_tol:
            eng_ok_s += 1

    stats['single_n20']['eng_ok'] += eng_ok_s
    stats['single_n20']['bit_ok'] += bit_ok_s
    stats['single_n20']['hd_sum'] += hd_sum_s
    stats['single_n20']['deg_sum'] += deg_s
    stats['single_n20']['total'] += num_reads

    inst_result['single_eng'] = eng_ok_s
    inst_result['single_bit'] = bit_ok_s
    inst_result['single_hd'] = round(hd_sum_s / num_reads, 2)
    inst_result['single_deg'] = deg_s
    inst_result['single_opt_val'] = float(val_s)

    # ── 2. Concat 4×5 ──
    mat_c, opt_c = gen_concatenated_random_qubo(sa_n, sa_max_sub)
    val_c = float(opt_c @ mat_c @ opt_c)
    _, _, deg_c = find_opt_brute_force(mat_c)
    target_c = ''.join(str(int(x)) for x in opt_c)

    Q_dict_c = matrix_to_qubo_dict(mat_c)
    ss_c = sampler.sample_qubo(Q_dict_c, num_reads=num_reads, num_sweeps=num_sweeps)

    eng_ok_c = 0
    bit_ok_c = 0
    hd_sum_c = 0
    for sample, energy, _ in ss_c.data(['sample', 'energy', 'num_occurrences']):
        found = ''.join(str(sample[k]) for k in range(sa_n))
        hd = sum(1 for a, b in zip(target_c, found) if a != b)
        hd_sum_c += hd
        if found == target_c:
            bit_ok_c += 1
        if abs(energy - val_c) < energy_tol:
            eng_ok_c += 1

    stats['concat_4x5']['eng_ok'] += eng_ok_c
    stats['concat_4x5']['bit_ok'] += bit_ok_c
    stats['concat_4x5']['hd_sum'] += hd_sum_c
    stats['concat_4x5']['deg_sum'] += deg_c
    stats['concat_4x5']['total'] += num_reads

    inst_result['concat_eng'] = eng_ok_c
    inst_result['concat_bit'] = bit_ok_c
    inst_result['concat_hd'] = round(hd_sum_c / num_reads, 2)
    inst_result['concat_deg'] = deg_c
    inst_result['concat_opt_val'] = val_c

    all_results.append(inst_result)

    # 진행 상황 출력
    if (inst + 1) % 50 == 0:
        elapsed = time.perf_counter() - t0
        eta = elapsed / (inst + 1) * (num_instances - inst - 1)
        s_rate = 100 * stats['single_n20']['eng_ok'] / stats['single_n20']['total']
        c_rate = 100 * stats['concat_4x5']['eng_ok'] / stats['concat_4x5']['total']
        print(f"  [{inst+1:>4}/{num_instances}] {elapsed:>6.1f}s (ETA {eta:.0f}s)"
              f"  Single:{s_rate:>5.1f}%  Concat:{c_rate:>5.1f}%")

elapsed = time.perf_counter() - t0

# ─── 최종 저장 ───
with open(results_path, 'w') as f:
    json.dump({
        'params': {
            'n': sa_n, 'max_sub': sa_max_sub, 'coeff': 'uniform(-1,1)',
            'num_instances': num_instances, 'num_reads': num_reads,
            'num_sweeps': num_sweeps, 'energy_tol': energy_tol,
            'elapsed_s': round(elapsed, 1), 'status': 'complete',
        },
        'stats': {c: {k: v for k, v in s.items()} for c, s in stats.items()},
        'instances': all_results,
    }, f)

# ─── 최종 요약 ───
print(f"\n{'═' * 95}")
print(f"  총 {num_instances} 인스턴스 × 2 조건 × {num_reads} reads = "
      f"{num_instances * 2 * num_reads:,} SA 샘플 | {elapsed:.1f}s")
print(f"  저장: {results_path}")
print(f"{'═' * 95}")
print(f"{'조건':<16} {'Energy 성공률':>16} {'Bit 성공률':>16} {'Avg HD':>10} {'Avg Degeneracy':>16}")
print(f"{'─' * 95}")
for c in conditions:
    r = stats[c]
    eng_rate = 100 * r['eng_ok'] / r['total']
    bit_rate = 100 * r['bit_ok'] / r['total']
    avg_hd = r['hd_sum'] / num_instances
    avg_deg = r['deg_sum'] / num_instances
    print(f"{c:<16} {r['eng_ok']:>6}/{r['total']} ({eng_rate:>5.1f}%) "
          f"{r['bit_ok']:>6}/{r['total']} ({bit_rate:>5.1f}%) "
          f"{avg_hd:>9.1f} {avg_deg:>15.1f}")
print(f"{'─' * 95}")

═══ SA 실험: Single (n=20) vs Concat (4×5) ═══
  n=20, max_sub=5, coeff=uniform(-1,1)
  instances=500, reads=100, sweeps=1000
  energy_tol=1e-06
  저장 경로: /home/yideun/qubo_dataset/hardened_posiform/results/single_vs_concat_n20_sub5_uniform_inst500_sw1000.json

  [  50/500]  203.8s (ETA 1835s)  Single: 91.0%  Concat: 90.5%
  [ 100/500]  407.4s (ETA 1629s)  Single: 89.4%  Concat: 90.2%
  [ 150/500]  610.8s (ETA 1425s)  Single: 88.7%  Concat: 90.0%
  [ 200/500]  812.1s (ETA 1218s)  Single: 89.1%  Concat: 89.8%
  [ 250/500] 1013.5s (ETA 1014s)  Single: 89.8%  Concat: 89.2%
  [ 300/500] 1220.3s (ETA 814s)  Single: 90.4%  Concat: 88.4%
  [ 350/500] 1425.7s (ETA 611s)  Single: 90.7%  Concat: 88.4%
  [ 400/500] 1630.3s (ETA 408s)  Single: 90.8%  Concat: 88.7%
  [ 450/500] 1832.5s (ETA 204s)  Single: 91.0%  Concat: 88.8%
  [ 500/500] 2033.8s (ETA 0s)  Single: 91.0%  Concat: 88.7%

═══════════════════════════════════════════════════════════════════════════════════════════════
  총 500 인스턴스 × 2 조건 ×

## 인스턴스별 분포 분석

In [10]:
# 인스턴스별 energy 성공률 분포
single_eng_rates = [r['single_eng'] / num_reads * 100 for r in all_results]
concat_eng_rates = [r['concat_eng'] / num_reads * 100 for r in all_results]

print("=== 인스턴스별 Energy 성공률 분포 ===")
print(f"{'':>20} {'Mean':>8} {'Std':>8} {'Min':>8} {'Median':>8} {'Max':>8}")
print(f"{'Single n=20':>20} {np.mean(single_eng_rates):>7.1f}% {np.std(single_eng_rates):>7.1f}% "
      f"{np.min(single_eng_rates):>7.1f}% {np.median(single_eng_rates):>7.1f}% {np.max(single_eng_rates):>7.1f}%")
print(f"{'Concat 4×5':>20} {np.mean(concat_eng_rates):>7.1f}% {np.std(concat_eng_rates):>7.1f}% "
      f"{np.min(concat_eng_rates):>7.1f}% {np.median(concat_eng_rates):>7.1f}% {np.max(concat_eng_rates):>7.1f}%")

# 축퇴도 분포
single_degs = [r['single_deg'] for r in all_results]
concat_degs = [r['concat_deg'] for r in all_results]
print(f"\n=== 축퇴도 분포 ===")
print(f"{'':>20} {'Mean':>8} {'Std':>8} {'Min':>8} {'Median':>8} {'Max':>8}")
print(f"{'Single n=20':>20} {np.mean(single_degs):>8.1f} {np.std(single_degs):>8.1f} "
      f"{np.min(single_degs):>8} {np.median(single_degs):>8.1f} {np.max(single_degs):>8}")
print(f"{'Concat 4×5':>20} {np.mean(concat_degs):>8.1f} {np.std(concat_degs):>8.1f} "
      f"{np.min(concat_degs):>8} {np.median(concat_degs):>8.1f} {np.max(concat_degs):>8}")

# 100% 성공 인스턴스 비율
s_perfect = sum(1 for r in single_eng_rates if r == 100)
c_perfect = sum(1 for r in concat_eng_rates if r == 100)
print(f"\n=== 100% Energy 성공 인스턴스 비율 ===")
print(f"  Single n=20: {s_perfect}/{num_instances} ({100*s_perfect/num_instances:.1f}%)")
print(f"  Concat 4×5:  {c_perfect}/{num_instances} ({100*c_perfect/num_instances:.1f}%)")

=== 인스턴스별 Energy 성공률 분포 ===
                         Mean      Std      Min   Median      Max
         Single n=20    91.0%    15.7%    23.0%   100.0%   100.0%
          Concat 4×5    88.7%    17.0%    30.0%    99.0%   100.0%

=== 축퇴도 분포 ===
                         Mean      Std      Min   Median      Max
         Single n=20      1.0      0.0        1      1.0        1
          Concat 4×5      1.0      0.0        1      1.0        1

=== 100% Energy 성공 인스턴스 비율 ===
  Single n=20: 268/500 (53.6%)
  Concat 4×5:  240/500 (48.0%)
